makes yields from the signal and background. Writes to a new file.

In [1]:
import ROOT
import numpy as np
import math
import pandas as pd
import fastjet
import matplotlib.pyplot as plt
import os
import ctypes
import array
import vector

In [2]:
# opening the file
# CHECK LOCATION FROM LAST CELL OF merge_hists.ipynb

#filepath = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/per_batch/merged_all_batches.root"
#filepath = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/test_merged_all_batches.root"
#filepath = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/per_batch/merged_all_batches.root"

#signal_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/merged/merged_signals.root"
#background_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/merged/Merged_Backgrounds.root"

ROOT.gDirectory.Clear()

In [ ]:
# 3mb high Nch
signal_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_nch60/Merged_Signals.root"
background_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_nch60/Merged_Backgrounds.root"

ROOT.gDirectory.Clear()

In [3]:
# 3mb izpc

signal_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_izpc/Merged_Signals.root"
background_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_izpc/Merged_Backgrounds.root"

ROOT.gDirectory.Clear()

In [ ]:
# 0mb high Nch
signal_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/0mb_nch60/Merged_Signals.root"
background_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/0mb_nch60/Merged_Backgrounds.root"

ROOT.gDirectory.Clear()

In [18]:
# 0mb inclusive
signal_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/0mb_inclusive/Merged_Signals.root"
background_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/0mb_inclusive/Merged_Backgrounds.root"

ROOT.gDirectory.Clear()

In [11]:
# 3mb inclusive
signal_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_inclusive/Merged_Signals.root"
background_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_inclusive/Merged_Backgrounds.root"

ROOT.gDirectory.Clear()

In [4]:
analysis_bins = [ [0,25], [25,36], [36,48], [48,60], [60,71], [71,78], [78,91], [91,97], [97,1000] ]

In [5]:
def makeYields(signal_path, background_path):
    '''
    takes a filepath and makes the yields
    returns:
    - yields per mult bin
    '''

    sig_file = ROOT.TFile.Open(signal_path, "READ")
    bkg_file = ROOT.TFile.Open(background_path, "READ")

    wta_yields = {}
    std_yields = {}
    y_axis_title = "#frac{1}{N_{trig}} #frac{d^{2}N^{pair}}{d#Delta#phi*#eta*}"

    for mult_bin in analysis_bins:
        bin_key = f"{mult_bin[0]} <  Nch < {mult_bin[1]}"

        # open histograms
        hSig_wta = sig_file.Get(f"WTA_sig_{mult_bin[0]}_{mult_bin[1]}")
        hSig_std = sig_file.Get(f"STD_sig_{mult_bin[0]}_{mult_bin[1]}")
        
        hBkg_wta = bkg_file.Get(f"WTA_bkg_{mult_bin[0]}_{mult_bin[1]}")
        hBkg_std = bkg_file.Get(f"STD_bkg_{mult_bin[0]}_{mult_bin[1]}")

        # get num jets
        num_jets_wta = sig_file.Get(f"num_jets_WTA_{mult_bin[0]}_{mult_bin[1]}").GetVal()
        num_jets_std = sig_file.Get(f"num_jets_STD_{mult_bin[0]}_{mult_bin[1]}").GetVal()

        # clone signals
        h_yield_wta = hSig_wta.Clone(f"WTA_yield_{mult_bin[0]}_{mult_bin[1]}")
        h_yield_std = hSig_std.Clone(f"STD_yield_{mult_bin[0]}_{mult_bin[1]}")

        # detach
        h_yield_wta.SetDirectory(0)
        h_yield_std.SetDirectory(0)
        
        # name axes
        h_yield_wta.SetTitle(f"WTA Yield ({bin_key});#Delta#eta*;#Delta#phi*;{y_axis_title}")
        h_yield_std.SetTitle(f"Standard Yield ({bin_key});#Delta#eta*;#Delta#phi*;{y_axis_title}")


        # find B(0,0)
        bin_zero_wta = hBkg_wta.FindBin(0.0, 0.0)
        bin_zero_std = hBkg_std.FindBin(0.0, 0.0)

        b00_wta = hBkg_wta.GetBinContent(bin_zero_wta)
        b00_std = hBkg_std.GetBinContent(bin_zero_std)

        # signal / background
        h_yield_wta.Divide(hBkg_wta)
        h_yield_std.Divide(hBkg_std)

        if num_jets_wta > 0:
            h_yield_wta.Scale(b00_wta / num_jets_wta)
        
        if num_jets_std > 0: 
            h_yield_std.Scale(b00_std / num_jets_std)

        wta_yields[bin_key] = h_yield_wta
        std_yields[bin_key] = h_yield_std

        print(f"{bin_key}")
        print(f"wta b00: {b00_wta}")
        print(f"std b00: {b00_std}")
        print(f"wta num_jets: {num_jets_wta}")
        print(f"std num_jets: {num_jets_std}")

    sig_file.Close()
    bkg_file.Close()
    
    return wta_yields, std_yields

In [6]:
def get_avg_Nch(signal_path):
    '''
    does what it says on the box
    '''
    wta_avg_Nch_dict = {}
    std_avg_Nch_dict = {}

    file = ROOT.TFile.Open(signal_path, "READ")

    for mult_bin in analysis_bins:
        bin_key = f"{mult_bin[0]} <  Nch < {mult_bin[1]}"
        # read avg Nch parameters
        #avg_Nch_wta = file.Get(f"avg_Nch_WTA_{mult_bin[0]}_{mult_bin[1]}").GetVal()
        #avg_Nch_std = file.Get(f"avg_Nch_STD_{mult_bin[0]}_{mult_bin[1]}").GetVal()

        total_Nch_wta = file.Get(f"total_Nch_WTA_{mult_bin[0]}_{mult_bin[1]}").GetVal()
        total_Nch_std = file.Get(f"total_Nch_STD_{mult_bin[0]}_{mult_bin[1]}").GetVal()

        num_jets_wta = file.Get(f"num_jets_WTA_{mult_bin[0]}_{mult_bin[1]}").GetVal()
        num_jets_std = file.Get(f"num_jets_STD_{mult_bin[0]}_{mult_bin[1]}").GetVal()

        if num_jets_wta > 0:
            avg_Nch_wta = total_Nch_wta / num_jets_wta
        else:
            avg_Nch_wta = 0
        
        if num_jets_std > 0:
            avg_Nch_std = total_Nch_std / num_jets_std
        else:
            avg_Nch_std = 0

        wta_avg_Nch_dict[bin_key] = avg_Nch_wta
        std_avg_Nch_dict[bin_key] = avg_Nch_std

    return wta_avg_Nch_dict, std_avg_Nch_dict


In [7]:
# make yields, get avg Nch values

wta_yields, std_yields = makeYields(signal_path, background_path)
wta_avg_Nch_dict, std_avg_Nch_dict = get_avg_Nch(signal_path)

0 <  Nch < 25
wta b00: 0.0
std b00: 0.0
wta num_jets: 0.0
std num_jets: 0.0
25 <  Nch < 36
wta b00: 0.0
std b00: 0.0
wta num_jets: 0.0
std num_jets: 0.0
36 <  Nch < 48
wta b00: 0.0
std b00: 0.0
wta num_jets: 0.0
std num_jets: 0.0
48 <  Nch < 60
wta b00: 1192.0
std b00: 1146.0
wta num_jets: 2.0
std num_jets: 2.0
60 <  Nch < 71
wta b00: 539489878.0
std b00: 671630684.0
wta num_jets: 5729475.0
std num_jets: 5729475.0
71 <  Nch < 78
wta b00: 120024072.0
std b00: 151547474.0
wta num_jets: 900473.0
std num_jets: 900473.0
78 <  Nch < 91
wta b00: 64425886.0
std b00: 82060176.0
wta num_jets: 372515.0
std num_jets: 372515.0
91 <  Nch < 97
wta b00: 6705448.0
std b00: 8595938.0
wta num_jets: 28801.0
std num_jets: 28801.0
97 <  Nch < 1000
wta b00: 4220682.0
std b00: 5449540.0
wta num_jets: 14810.0
std num_jets: 14810.0


In [8]:
# save output

#EDIT FILE PATH IF NECESSARY

#out_file = ROOT.TFile("/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_nch60/merged_yields.root", "RECREATE")
#out_file = ROOT.TFile("/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/0mb_nch60/merged_yields.root", "RECREATE")
#out_file = ROOT.TFile("/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/0mb_inclusive/merged_yields.root", "RECREATE")
#out_file = ROOT.TFile("/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_inclusive/merged_yields.root", "RECREATE")

out_file = ROOT.TFile("/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_izpc/merged_yields.root", "RECREATE")

for mult_bin in analysis_bins:
    bin_key = f"{mult_bin[0]} <  Nch < {mult_bin[1]}"

    # write histograms
    wta_yields[bin_key].Write()
    std_yields[bin_key].Write()

    # write TParams
    param_avg_Nch_wta = ROOT.TParameter('double')(f"avg_Nch_WTA_{mult_bin[0]}_{mult_bin[1]}", wta_avg_Nch_dict[bin_key])
    param_avg_Nch_std = ROOT.TParameter('double')(f"avg_Nch_STD_{mult_bin[0]}_{mult_bin[1]}", std_avg_Nch_dict[bin_key])
    param_avg_Nch_wta.Write()
    param_avg_Nch_std.Write()

out_file.Close()